# 09 • Indicadores e segmentação de clientes

## Preparar esta sessão
Execute as células abaixo antes de começar a aula. Elas localizam os arquivos e preparam os objetos necessários. Você não precisa decorar essa preparação agora.

In [ ]:
from pathlib import Path
import pandas as pd
import math
BASE = Path.cwd()
if not (BASE / 'dados').exists() and (BASE.parent / 'dados').exists():
    BASE = BASE.parent
DADOS = BASE / 'dados'
SAIDAS = BASE / 'resultados'
SAIDAS.mkdir(exist_ok=True)
assert (DADOS/'penguins.csv').exists(), 'Descompacte o curso inteiro e abra na pasta correta.'


## Preparação independente
Execute esta célula para recriar os objetos necessários dos módulos anteriores.

In [ ]:
p = pd.read_csv(DADOS / 'penguins.csv')
print(p.head(3).to_string(index=False))
print('Dimensões:', p.shape)
print('Colunas:', p.columns.tolist())
print(type(p['species']).__name__)

mascara = p['species'].eq('Adelie') & p['body_mass_g'].gt(4000)
print(p.loc[mascara, ['species', 'island', 'body_mass_g']].head())
print(p.iloc[:3, :2])

qualidade = pd.DataFrame({'ausentes':p.isna().sum(), 'percentual':p.isna().mean().mul(100)})
print(qualidade)
medidos = p.dropna(subset=['body_mass_g']).copy()
print('Observações com massa:', len(medidos))

enriquecido = p.assign(body_mass_kg=p['body_mass_g']/1000)
print(enriquecido[['body_mass_g','body_mass_kg']].head())
print(p['species'].astype('string').str.lower().head())

resumo = p.groupby('species', as_index=False, observed=True).agg(
    observacoes=('body_mass_g','size'),
    massas_validas=('body_mass_g','count'),
    massa_media_g=('body_mass_g','mean'))
print(resumo)

nomes = pd.DataFrame({'species':['Adelie','Chinstrap','Gentoo'],
                      'nome_pt':['Adélia','Barbicha','Papua']})
cruzado = p.merge(nomes, on='species', how='left', validate='many_to_one', indicator=True)
print(cruzado['_merge'].value_counts())

r = pd.read_csv(DADOS/'online_retail.csv', dtype={
    'InvoiceNo':'string','StockCode':'string','CustomerID':'string'})
r['InvoiceDate'] = pd.to_datetime(r['InvoiceDate'], errors='coerce')
cancelada = r['InvoiceNo'].str.upper().str.startswith('C', na=False)
elegivel = (r['Quantity'].gt(0) & r['UnitPrice'].gt(0)
            & r['InvoiceDate'].notna() & ~cancelada)
v = r.loc[elegivel].copy()
v['valor'] = v['Quantity'] * v['UnitPrice']
print('Linhas brutas:',len(r),'Linhas elegíveis:',len(v))
print(r.isna().sum())

mensal = v.set_index('InvoiceDate')['valor'].resample('MS').sum(min_count=1)
print(mensal)
print('Cobertura:',r['InvoiceDate'].min(),r['InvoiceDate'].max())

diario = v.groupby(v['InvoiceDate'].dt.normalize())['valor'].sum()
diario = diario.reindex(pd.date_range(diario.index.min(),diario.index.max(),freq='D'), fill_value=0)
print(diario.head())
# Hipótese: dias não registrados dentro da cobertura tiveram zero venda elegível.

### Fazendo o RFM sem decorar a sigla

Para cada cliente, queremos três respostas: há quantos dias ele comprou, em quantas faturas apareceu e quanto comprou na visão elegível. groupby('CustomerID') define uma linha por cliente. max da data encontra a última compra; nunique da fatura conta compras distintas; sum do valor calcula o total. A data de referência fica um dia após a última venda observada, para manter o exercício reproduzível.

Exemplo fictício: referência em 10/01; compras em 02/01 e 08/01, em duas faturas de £20 e £30. Recência = 2 dias, frequência = 2 e monetário = £50. Comprar vários itens na mesma fatura não aumenta a frequência. Baixa recência significa compra mais recente; alto monetário significa maior valor observado. Não são três medidas com a mesma direção de interpretação.

Ao calcular participação acumulada, ordene o monetário do maior para o menor. cumsum acumula os valores até cada linha; dividir pelo total transforma a soma acumulada em proporção. O último valor deve estar próximo de 1, isto é, 100%. Isso descreve concentração histórica; não é garantia de receita futura.

### Aprofundamento e cuidados

Um indicador precisa de nome, fórmula, unidade, período, população e exclusões. Receita por cliente mede valor, frequência mede número de faturas e recência mede dias desde a última compra. Este é um RFM descritivo: não vamos chamar automaticamente esses números de previsão de abandono ou de valor futuro do cliente.

Para evitar datas que mudam todos os dias, a referência será o dia seguinte à última data de venda elegível. Frequência deve contar faturas distintas, não itens. Clientes ausentes ficam fora da segmentação e sua participação na receita é reportada. Cada observação de RFM representa um cliente.

Ticket médio global é total dividido por quantidade de pedidos. A média dos tickets médios por país dá peso igual a países com volumes diferentes e em geral responde a outra pergunta. Escreva a fórmula antes de escolher mean ou sum. Uma proporção deve preservar seu denominador e uma divisão precisa tratar zero.


## Exemplo guiado

In [ ]:
identificados = v.dropna(subset=['CustomerID']).copy()
referencia = v['InvoiceDate'].max().normalize() + pd.Timedelta(days=1)
rfm = identificados.groupby('CustomerID').agg(
    ultima_compra=('InvoiceDate','max'), frequencia=('InvoiceNo','nunique'), monetario=('valor','sum'))
rfm['recencia_dias'] = (referencia-rfm['ultima_compra'].dt.normalize()).dt.days
print(rfm.head())

## Exercício 09.1
Mostre dez clientes com maior valor monetário.

In [ ]:
# Escreva sua resposta aqui


## Exercício 09.2
Calcule a porcentagem da receita elegível sem identificação de cliente.

In [ ]:
# Escreva sua resposta aqui


## Exercício 09.3
Selecione clientes com recência até 30 dias e pelo menos cinco faturas. A regra é didática, não um modelo validado.

In [ ]:
# Escreva sua resposta aqui


## Exercício 09.4
Calcule a participação acumulada dos clientes na receita identificada, em ordem decrescente.

In [ ]:
# Escreva sua resposta aqui


## Exercício 09.5
Valide que a soma monetária por cliente coincide com o total identificado.

In [ ]:
# Escreva sua resposta aqui
